### PyTorch's tensor library

In [1]:
import torch

In [2]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2])
tensor2d = torch.tensor([[1, 2], [3, 4]])
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

In [3]:
# 32-bit precision for deep learning, GPU architectures designed for this and better efficiency overall
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

torch.float32


In [4]:
tensor2d.shape

torch.Size([2, 2])

In [5]:
# tensor2d.reshape(1, 4) -- less common than view
tensor2d.view(1, 4)

tensor([[1, 2, 3, 4]])

In [6]:
tensor2d.shape

torch.Size([2, 2])

In [7]:
tensor2d.T # transpose a tensor, flip it across its diagonal

tensor([[1, 3],
        [2, 4]])

In [8]:
tensor2d.matmul(tensor2d.T)
# tensor2d @ tensor2d.T accomplishes same thing but less common

tensor([[ 5, 11],
        [11, 25]])

In [9]:
# see https://pytorch.org/docs/stable/tensors.html for more operations

### PyTorch's automatic differentiation engine

- a *computation graph* is a directed graph that allows us to express and visualize mathematical expressions

- it lays out the sequence of calculations needed to compute the output of a neural network

In [10]:
# example of a forward pass (prediction step) of a simple logistic regression classifier that returns a score between 0 and 1
# that is compared to the true class label when computing the loss

import torch.nn.functional as F

y = torch.tensor([1.0]) # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # the weight parameter
b = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


PyTorch's autograd engine constructions a computation graph in the background by tracking every operation performed on tensors. Then, calling the grad function, we can compute the gradient of the loss with respect to model parameter w1:

In [11]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True) # by default, PyTorch destroys the computation graph after calculating the gradients to free memory
grad_L_b = grad(loss, b, retain_graph=True)

In [12]:
# resulting values of the loss with respect to the model's parameters:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [13]:
# PyTorch provides even more high-level tools to compute the gradients of all the leaf nodes in the graph

loss.backward()

In [14]:
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


### Implementing multilayer neural networks

Implementing a neural network in PyTorch is typically done by subclassing the torch.nn.Module class to define a custom network architecture. The class allows for encapsulation of layers and operations and keeping track of a model's parameters.

- the __init__ constructor is where we define the network layers

- the forward method describes how the input data passes through the network and comes together as a computation graph

- the backward method is used during training to compute gradients of the loss function with respect to the model parameters (do not have to implement typically)

In [15]:
# classic multilayer perceptron with two hidden layers

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30), 
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs), 
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

        

In [16]:
model = NeuralNetwork(50, 3)

In [17]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [18]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [19]:
print(model.layers[0].weight)
# print(model.layers[0].bias)

Parameter containing:
tensor([[-0.0206, -0.0263, -0.0136,  ...,  0.0244,  0.0623, -0.0409],
        [-0.0550, -0.0591, -0.1050,  ..., -0.0528,  0.0345,  0.0206],
        [ 0.0091, -0.0817, -0.0440,  ..., -0.0863,  0.0233, -0.0826],
        ...,
        [-0.0169,  0.1393,  0.1326,  ...,  0.0014, -0.0394, -0.0666],
        [-0.0525,  0.0898,  0.0547,  ...,  0.0682,  0.0027, -0.1014],
        [ 0.1236,  0.0153,  0.0470,  ...,  0.0349, -0.1358, -0.0751]],
       requires_grad=True)


In [20]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [21]:
torch.manual_seed(42) # model weights are initialized with small random numbers in order to break symmetry during training, 
# otherwise the nodes would just perform the same operations and updates during backprop and not allow the model to learn mappnigs from input to output

X = torch.rand((1, 50))
out = model(X) # execute the forward pass of the model (calculating the output tensors from the input tensors)

print(out)

tensor([[-0.1487,  0.3100, -0.0593]], grad_fn=<AddmmBackward0>)


- The three numbers returned represent the score assigned to each of the three output nodes
- The grad_fn represents the last-used function to compute a variable in the computation graph
- AddmmBackward0 means the tensor was created via a matrix multiplication (mm) and an addition operation (Add) 
- PyTorch uses this info when it computes gradients during backprop
- Best practice is to use torch.no_grad() when using a model for inference since constructing the computational graph can be wasteful (tells PyTorch it does not need to keep track of the gradients)

In [22]:
with torch.no_grad():
    out = model(X)

print(out)

tensor([[-0.1487,  0.3100, -0.0593]])


Note that it is common practice to code models such that they return the outputs of the last layer without passing them to a nonlinear activation function - computing class-membership probabilities for our predictions can be done explicitly:

In [26]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1) # values can now be interpreted as class-membership probabilities that add up to 1

print(out)

tensor([[0.2721, 0.4304, 0.2975]])


### Setting up efficient data loaders

- PyTorch implements a Dataset and a DataLoader class
- The Dataset class is used to instantiate objects that define how each data record is loaded
- The DataLoader class handles how data is shuffled and assembled into batches

In [27]:
# sample dataset with two features

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

In [28]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])

y_test = torch.tensor([0, 1])

In [ ]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X # could be file paths, file objects, database connectors etc
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
# now we can use the DataLoader class to sample from it:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0, # when not set to 0, multiple worker processes are launched to load data in parallel
    drop_last=True # good practice so a potentially substantially smaller batch will not disturb convergence during training
)

In [35]:
test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [36]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

# train loader iterates over the training dataset visiting each training example exactly once (known as a training epoch)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


### A typical training loop

In [ ]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2) # two input features and two class labels
optimizer = torch.optim.SGD(model.parameters(), lr=0.5) # learning rate is a hyperparameter to tune

num_epochs = 3 # also a hyperparameter

for epoch in range(num_epochs):
    model.train() # setting to put model into training mode, particularly for dropout and batch normalization layers which behave differently at inference time
    
    for batch_idx, (features, labels) in enumerate(train_loader): # iterate over each batch
        
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function that applies softmax function internally

        optimizer.zero_grad() # resets gradients of all model parameters before computing new backward pass
        loss.backward() # calculate the gradients in the computaton graph
        optimizer.step() # use the gradients to update the model parameters to minimize the loss
        # SGD optimizer means multiplying the gradients with the learning rate and adding the scaled negative gradient to the parameters

        ## Logging
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
        

/Users/jenniferjordache/Developer/core-knowledge-ml/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [39]:
model.eval() # setting to put model into evaluation mode and make predictions

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [40]:
# obtain class membership probabilties
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [ ]:
# example 1 has 99.91% probability of belonging to class 0 and 0.09% probability of belonging to class 1
# convert probabilites to class label predictons using argmax

predictions = torch.argmax(probas, dim=1) # could also apply argmax to logits directly
print(predictions)

tensor([0, 0, 0, 1, 1])


In [42]:
torch.sum(y_train == predictions)

tensor(5)

Function to compute accuracy more generally

In [45]:
def compute_accuracy(model, dataloader):
    model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader): # iterate through each batch, but only once across all training data
        with torch.no_grad():
            logits = model(features)
        
        predictions = torch.argmax(logits, dim=1)
        compare = predictions == labels
        correct += torch.sum(compare)
        total_examples += len(compare)
    
    return (correct / total_examples).item()
        

In [46]:
compute_accuracy(model, train_loader)

1.0

In [47]:
compute_accuracy(model, test_loader)

1.0

### Saving and loading models

In [48]:
# the model's state_dict is a dictionary object 
torch.save(model.state_dict(), "model.pth")

In [49]:
# restore from disk
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model.pth", weights_only=True))

/Users/jenniferjordache/Developer/core-knowledge-ml/myenv/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


<All keys matched successfully>

### Some notes on distributed training

##### DistributedDataParallel (DDP) strategy in Pytorch

- As an example, when you have two GPUs, each will receive a copy of the model. During every training iteration, each model will receive a batch from the data loader (we can use a DistributedSampler to ensure each GPU will receive a different, non-overlapping batch when using DDP)
- Since each model copy will see a different sample of the training data, the model copies will return different logits as outputs and compute different gradients during backward pass
- These gradients are then averaged and synchronized during training to update the models (so the models don't diverge)